# Phase A — Forward 1D Poisson PIFT (Colab edition)

Self-contained walkthrough of the **Physics-Informed Information Field Theory** sampler from Alberts & Bilionis (2023, *J. Comput. Phys.* 486:112100), specialized to the 1D Poisson example from Section 5.1 of the paper.

We solve the inverse problem of inferring a scalar field φ(x) on [0, 1] from noisy measurements, with the physics prior

$$U[\varphi] \;=\; \tfrac{1}{2}\,\mathbb{E}_x\!\left[\,(-\varphi''(x) - f(x))^{2}\,\right]$$

and the Boltzmann posterior $\,p(\varphi\mid d) \propto \exp(-\beta\,U[\varphi])\,p(d\mid\varphi)$, sampled via SGLD on the sine-basis coefficients.

**No local setup required** — everything installs from GitHub. GPU recommended (Runtime → Change runtime type → GPU).

## 1. Install & imports
Pulls the package directly from GitHub plus `tqdm` for progress feedback. Restart of the runtime is **not** required.

In [ ]:
%pip install -q git+https://github.com/cmhobbs96/pift-od-il-inverse-problems.git
%pip install -q tqdm

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import clear_output

jax.config.update('jax_enable_x64', True)
print('JAX backend:', jax.default_backend(), '| devices:', jax.devices())

from core.parameterizations import SineBasisField
from core.energies import poisson_residual_energy
from core.likelihoods import gaussian_nll
from core.reference_solver import solve_poisson_dirichlet_fd
from utils.diagnostics import effective_sample_size_bulk, credible_interval_coverage

## 2. CONFIG
Single dict, no YAML, no hidden defaults. Tweak in place and re-run from Cell 3 onward.

In [ ]:
CONFIG = {
    # Problem -------------------------------------------------------------
    'seed':       7,        # PRNG seed for observations + sampler
    'n_obs':      28,       # number of noisy measurements of phi(x)
    'noise_std':  0.08,     # std of additive Gaussian observation noise

    # Field representation -----------------------------------------------
    'n_modes':    12,       # number of sine-basis modes (zero Dirichlet BCs)
    'n_quad':     96,       # quadrature points per SGLD step (resampled each step)
    'n_grid':     300,      # plotting / FD grid resolution

    # PIFT temperature ----------------------------------------------------
    'beta':       12.0,     # physics trust parameter (β → ∞ collapses to MAP)

    # SGLD sampler --------------------------------------------------------
    'n_steps':    16000,    # total SGLD iterations
    'burn_in':    4000,     # discarded as warm-up
    'thin':       8,        # keep every k-th post-burn-in sample
    'step_size0': 2e-3,     # initial step size (alpha_0)
    'decay':      0.55,     # polynomial decay exponent: alpha_t = alpha_0 / (1+t)^decay
    'max_cond':   100.0,    # cap on preconditioner condition number

    # Live feedback -------------------------------------------------------
    'plot_every': 1000,     # redraw posterior every N steps during sampling
}

## 3. Problem setup & FD reference
Ground-truth field $\varphi^\star(x) = \sin(\pi x) + 0.35\sin(2\pi x)$ with the matching forcing $f = -\varphi^{\star\prime\prime}$. We compute a finite-difference reference solution as a sanity check before running PIFT.

In [ ]:
def phi_true(x):
    return jnp.sin(jnp.pi * x) + 0.35 * jnp.sin(2.0 * jnp.pi * x)

def forcing(x):
    return (jnp.pi**2) * jnp.sin(jnp.pi * x) + 0.35 * (2.0 * jnp.pi)**2 * jnp.sin(2.0 * jnp.pi * x)

key = jax.random.PRNGKey(CONFIG['seed'])
key, x_key, n_key = jax.random.split(key, 3)
x_obs = jnp.sort(jax.random.uniform(x_key, shape=(CONFIG['n_obs'],), minval=0.03, maxval=0.97))
y_clean = phi_true(x_obs)
y_obs = y_clean + CONFIG['noise_std'] * jax.random.normal(n_key, shape=(CONFIG['n_obs'],))

x_grid = np.linspace(0.0, 1.0, CONFIG['n_grid'])
phi_truth_grid = np.asarray(phi_true(jnp.asarray(x_grid)))

# FD reference
x_fd, phi_fd = solve_poisson_dirichlet_fd(
    forcing_fn=lambda z: np.asarray(forcing(jnp.asarray(z)), dtype=float),
    n_points=CONFIG['n_grid'], domain=(0.0, 1.0), bc=(0.0, 0.0),
)
fd_l2 = float(np.sqrt(np.mean((phi_fd - phi_truth_grid)**2)))
print(f'FD reference L2 vs analytical truth: {fd_l2:.2e}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_grid, phi_truth_grid, 'k-', lw=2, label='truth')
ax.plot(x_fd, phi_fd, 'b--', lw=1.5, label='FD reference')
ax.scatter(np.asarray(x_obs), np.asarray(y_obs), c='red', s=30, zorder=5, label='obs')
ax.set_xlabel('x'); ax.set_ylabel(r'$\varphi(x)$'); ax.legend(); ax.set_title('Setup'); plt.show()

## 4. PIFT sampling with live plot
Implements Algorithm 1 from Alberts & Bilionis: SGLD on the sine-basis coefficients $\theta$, with the joint Hamiltonian $H(\theta) = -\log p(d\mid\theta) + \beta\,U[\theta]$. The quadrature points for $U[\theta]$ are **resampled each step** (stochastic estimator). The diagonal preconditioner normalizes curvature across modes so high-frequency components don't dominate the step size.

We warm-start $\theta_0$ from a least-squares projection of the FD reference onto the sine basis, then run SGLD with a `tqdm` progress bar and a live posterior plot every `plot_every` steps.

In [ ]:
field = SineBasisField(CONFIG['n_modes'])
obs_matrix = field.design_matrix(x_obs)
basis_grid = np.asarray(field.design_matrix(jnp.asarray(x_grid)))

# Warm-start from FD projection
B_init = np.asarray(field.design_matrix(jnp.asarray(x_fd)))
theta = jnp.asarray(np.linalg.lstsq(B_init, np.asarray(phi_fd), rcond=None)[0])

# Diagonal preconditioner (mode-wise curvature normalization)
precond = field.preconditioner(beta=CONFIG['beta'])
floor = jnp.max(precond) / CONFIG['max_cond']
precond = jnp.clip(precond, min=floor); precond = precond / jnp.max(precond)
sqrt_precond = jnp.sqrt(precond)

beta = float(CONFIG['beta']); noise_std = float(CONFIG['noise_std']); n_quad = int(CONFIG['n_quad'])

@jax.jit
def grad_step(theta, key, alpha_t):
    key, qk, nk = jax.random.split(key, 3)
    x_quad = jax.random.uniform(qk, shape=(n_quad,), minval=0.0, maxval=1.0)
    phys_e, phys_g, _ = poisson_residual_energy(theta, x_quad, field, forcing)
    like_e, like_g, _ = gaussian_nll(theta, obs_matrix, y_obs, noise_std)
    grad = like_g + beta * phys_g
    noise = jax.random.normal(nk, shape=theta.shape, dtype=theta.dtype)
    theta_new = theta - alpha_t * precond * grad + jnp.sqrt(2.0 * alpha_t) * sqrt_precond * noise
    return theta_new, key, like_e + beta * phys_e

# Storage
n_steps = CONFIG['n_steps']
chain = np.zeros((n_steps, CONFIG['n_modes']))
ham_trace = np.zeros(n_steps)

key, sk = jax.random.split(key)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for t in tqdm(range(n_steps), desc='SGLD'):
    alpha_t = CONFIG['step_size0'] / ((1.0 + t) ** CONFIG['decay'])
    theta, sk, h = grad_step(theta, sk, alpha_t)
    chain[t] = np.asarray(theta)
    ham_trace[t] = float(h)

    if (t + 1) % CONFIG['plot_every'] == 0 or t == n_steps - 1:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        # Live posterior using samples available so far
        bi = min(CONFIG['burn_in'], t // 2)
        live_samples = chain[bi:t+1:CONFIG['thin']]
        if live_samples.shape[0] > 1:
            phi_s = live_samples @ basis_grid.T
            mean = phi_s.mean(0); lo = np.percentile(phi_s, 5, 0); hi = np.percentile(phi_s, 95, 0)
            axes[0].fill_between(x_grid, lo, hi, alpha=0.3, label='90% CI')
            axes[0].plot(x_grid, mean, 'b-', label='posterior mean')
        axes[0].plot(x_grid, phi_truth_grid, 'k--', lw=1.5, label='truth')
        axes[0].scatter(np.asarray(x_obs), np.asarray(y_obs), c='red', s=20, zorder=5, label='obs')
        axes[0].set_xlabel('x'); axes[0].set_ylabel(r'$\varphi(x)$')
        axes[0].set_title(f'Posterior (step {t+1}/{n_steps})'); axes[0].legend(loc='upper right', fontsize=8)
        axes[1].plot(ham_trace[:t+1], lw=0.5)
        axes[1].set_xlabel('step'); axes[1].set_ylabel('Hamiltonian'); axes[1].set_title('Energy trace')
        plt.tight_layout(); plt.show()

print(f'Sampling complete: {n_steps} steps, {(n_steps - CONFIG["burn_in"]) // CONFIG["thin"]} kept samples')

## 5. Results & diagnostics
Three numbers tell us whether the sampler did its job:
- **L2 error** of the posterior mean against the analytical truth (should be ~`noise_std / sqrt(n_obs)` order of magnitude).
- **90% credible-interval coverage** at the held-in observation locations — should be close to 0.9 if the posterior is well-calibrated.
- **Bulk effective sample size** per coefficient — measures sampler mixing.

In [ ]:
samples = chain[CONFIG['burn_in']::CONFIG['thin']]
phi_samples = samples @ basis_grid.T
phi_mean = phi_samples.mean(0); phi_std = phi_samples.std(0)
phi_lo = np.percentile(phi_samples, 5, 0); phi_hi = np.percentile(phi_samples, 95, 0)

l2 = float(np.sqrt(np.mean((phi_mean - phi_truth_grid)**2)))
max_err = float(np.max(np.abs(phi_mean - phi_truth_grid)))
cov_90 = credible_interval_coverage(phi_samples, phi_truth_grid, level=0.90)
ess = effective_sample_size_bulk(samples)

print(f'  L2 error      : {l2:.4f}')
print(f'  Max abs error : {max_err:.4f}')
print(f'  90% coverage  : {cov_90:.3f}  (target ~0.90)')
print(f'  ESS bulk min  : {ess.min():.1f}  / {samples.shape[0]} kept samples')
print(f'  FD reference L2 : {fd_l2:.2e}  (analytical baseline)')

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(x_grid, phi_lo, phi_hi, alpha=0.3, color='C0', label='PIFT 90% CI')
ax.plot(x_grid, phi_mean, 'C0-', lw=2, label='PIFT posterior mean')
ax.plot(x_grid, phi_truth_grid, 'k--', lw=1.5, label='analytical truth')
ax.plot(x_fd, phi_fd, 'g:', lw=1.5, label='FD reference')
ax.scatter(np.asarray(x_obs), np.asarray(y_obs), c='red', s=30, zorder=5, label='observations')
ax.set_xlabel('x'); ax.set_ylabel(r'$\varphi(x)$'); ax.legend(); ax.set_title('Phase A — Forward 1D Poisson PIFT'); plt.show()

## 6. Optional: Monte Carlo comparison
Random-walk Metropolis on the same posterior, for context. Far less efficient than SGLD on this problem because each proposal is global and the curvature mismatch across modes hurts acceptance — but it requires no gradients.

In [ ]:
def neg_log_posterior(theta, x_quad):
    pe, _, _ = poisson_residual_energy(theta, x_quad, field, forcing)
    le, _, _ = gaussian_nll(theta, obs_matrix, y_obs, noise_std)
    return float(le + beta * pe)

n_mc = 4000; proposal_std = 2e-3
theta_mc = jnp.asarray(np.linalg.lstsq(B_init, np.asarray(phi_fd), rcond=None)[0])
key, qk = jax.random.split(key)
x_quad_fixed = jax.random.uniform(qk, shape=(n_quad,), minval=0.0, maxval=1.0)
current_logp = neg_log_posterior(theta_mc, x_quad_fixed)
mc_chain = np.zeros((n_mc, CONFIG['n_modes'])); accepts = 0
rng = np.random.default_rng(CONFIG['seed'])
for i in tqdm(range(n_mc), desc='MC'):
    prop = theta_mc + proposal_std * rng.standard_normal(CONFIG['n_modes'])
    prop_logp = neg_log_posterior(prop, x_quad_fixed)
    if np.log(rng.random()) < current_logp - prop_logp:
        theta_mc = prop; current_logp = prop_logp; accepts += 1
    mc_chain[i] = np.asarray(theta_mc)
print(f'MC acceptance rate: {accepts/n_mc:.2%}')

mc_samples = mc_chain[n_mc // 2 :]
mc_phi = mc_samples @ basis_grid.T
mc_mean = mc_phi.mean(0); mc_lo = np.percentile(mc_phi, 5, 0); mc_hi = np.percentile(mc_phi, 95, 0)
mc_l2 = float(np.sqrt(np.mean((mc_mean - phi_truth_grid)**2)))
print(f'MC posterior L2: {mc_l2:.4f}  (PIFT/SGLD: {l2:.4f})')

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(x_grid, phi_lo, phi_hi, alpha=0.25, color='C0', label='PIFT 90% CI')
ax.plot(x_grid, phi_mean, 'C0-', lw=2, label=f'PIFT (L2={l2:.3f})')
ax.fill_between(x_grid, mc_lo, mc_hi, alpha=0.25, color='C1', label='MC 90% CI')
ax.plot(x_grid, mc_mean, 'C1-', lw=2, label=f'MC (L2={mc_l2:.3f})')
ax.plot(x_grid, phi_truth_grid, 'k--', lw=1.5, label='truth')
ax.scatter(np.asarray(x_obs), np.asarray(y_obs), c='red', s=20, zorder=5)
ax.set_xlabel('x'); ax.set_ylabel(r'$\varphi(x)$'); ax.legend(); ax.set_title('PIFT vs MC'); plt.show()